In [0]:
# 1. Define your credentials
storage_account_name = "shiwamdataproject01"
storage_account_key = "bNNQtSWcqFO6swEaxGWHRRJ2kZfYmo4bkgwcijVnoWmNeng+NSiFtZ2GPqwnMyX4LzIskpmgwKo2+ASt7y8uQg=="
container_name = "medallion" # The container you created in ADLS

# 2. Configure Spark to use the Access Key
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
    storage_account_key
)

# 3. Define the Base Path for your project
# Use 'abfss' (Azure Blob File System Driver - Secure)
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"

print(f"Connection set up for: {base_path}")

In [0]:
from pyspark.sql import functions as F

# 1. Use the base_path from our setup (or redefine it here)
storage_account_name = "shiwamdataproject01"
container_name = "medallion"
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
raw_path = f"{base_path}raw/transactions/"

# 2. Generate Data with SKEW (80% of data for store_id 100)
df = spark.range(0, 250000000) \
    .withColumn("transaction_id", F.expr("uuid()")) \
    .withColumn("amount", (F.rand() * 1000).cast("decimal(10,2)")) \
    .withColumn("event_time", F.current_timestamp()) \
    .withColumn("store_id", F.when(F.rand() < 0.8, 100).otherwise((F.rand() * 50).cast("int")))\
    .withColumn("customer_id", (F.rand() * 4 + 1).cast("int"))

# 3. Write as Parquet to the Landing Zone
df.write.format("parquet").mode("append").save(raw_path)
print("Batch 1: 250M rows landed in /raw/")

In [0]:
# dbutils.fs.rm("abfss://medallion@shiwamdataproject01.dfs.core.windows.net/raw/transactions/", recurse=True)